In [ ]:
import os
import csv
import torch
import random
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline, DDIMScheduler, AutoencoderKL
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
INPUT_DIR = "../Dataset/downloaded_images/"
OUTPUT_DIR = "../Dataset/image_synthetic_dataset"

# --- 2. THE FORENSIC DISGUISE POOLS ---
# We break these down into aggressive, highly descriptive elements
HAIR = [
    "completely bald shaved head", 
    "long messy unkempt hair covering the ears", 
    "short military buzz cut with a sharp hairline", 
    "voluminous curly wig altering the head shape", 
    "greasy shoulder-length hair"
]
HEADWEAR = [
    "dark winter beanie pulled very low over the eyebrows", 
    "baseball cap pulled down casting deep heavy shadows over the eyes", 
    "dark hoodie tightly cinched around the face", 
    "wide-brimmed fedora hat", 
    "tactical boonie hat"
]
EYEWEAR = [
    "thick-rimmed black glasses with lens glare", 
    "dark opaque aviator sunglasses obscuring the eyes", 
    "reflective silver sports goggles tight to the face"
]
LOWER_FACE = [
    "blue surgical medical mask covering the nose and mouth", 
    "dark tactical neck gaiter pulled up over the bridge of the nose", 
    "thick knit wool scarf tightly wrapping the jawline and mouth"
]
FACIAL_HAIR_MAKEUP = [
    "thick bushy heavy beard hiding the jawline", 
    "heavy theatrical contouring makeup altering bone structure", 
    "heavy 5-o-clock shadow and unkempt appearance"
]
ENVIRONMENT = [
    "grainy CCTV security camera footage with green night vision", 
    "sharp 90-degree side profile view", 
    "harsh overhead interrogation room lighting casting deep shadows"
]

def build_disguise_tasks():
    """Builds 10 aggressive combinations guaranteeing Upper Occlusion or Hair in every prompt."""
    tasks = []
    
    # 1. Hair + Eyewear
    tasks.append({"prompt": f"{random.choice(HAIR)} and wearing {random.choice(EYEWEAR)}", "scale": 0.55})
    # 2. Headwear + Eyewear (Heavy Upper T-Zone block)
    tasks.append({"prompt": f"wearing a {random.choice(HEADWEAR)} and {random.choice(EYEWEAR)}", "scale": 0.50})
    # 3. Hair + Lower Face Mask
    tasks.append({"prompt": f"{random.choice(HAIR)} and wearing a {random.choice(LOWER_FACE)}", "scale": 0.45})
    # 4. Headwear + Lower Face Mask (Top & Bottom occlusion)
    tasks.append({"prompt": f"wearing a {random.choice(HEADWEAR)} and a {random.choice(LOWER_FACE)}", "scale": 0.40})
    # 5. Hair + Structural Alteration (Beard/Makeup)
    tasks.append({"prompt": f"{random.choice(HAIR)} and has {random.choice(FACIAL_HAIR_MAKEUP)}", "scale": 0.50})
    # 6. Headwear + Structural Alteration (Beard/Makeup)
    tasks.append({"prompt": f"wearing a {random.choice(HEADWEAR)} and has {random.choice(FACIAL_HAIR_MAKEUP)}", "scale": 0.50})
    # 7. Extreme Combo: Headwear + Eyewear + Mask (The "Bank Robber")
    tasks.append({"prompt": f"wearing a {random.choice(HEADWEAR)}, {random.choice(EYEWEAR)}, and a {random.choice(LOWER_FACE)}", "scale": 0.35})
    # 8. Hair + Headwear (e.g. long hair sticking out from a beanie)
    tasks.append({"prompt": f"{random.choice(HAIR)} sticking out from under a {random.choice(HEADWEAR)}", "scale": 0.55})
    # 9. Hair + Eyewear + Environment (Angle/Lighting shift)
    tasks.append({"prompt": f"{random.choice(HAIR)} and wearing {random.choice(EYEWEAR)}, {random.choice(ENVIRONMENT)}", "scale": 0.50})
    # 10. Headwear + Environment (Angle/Lighting shift)
    tasks.append({"prompt": f"wearing a {random.choice(HEADWEAR)}, {random.choice(ENVIRONMENT)}", "scale": 0.55})
    
    random.shuffle(tasks)
    return tasks

def setup_sd_pipeline(device, dtype):
    tqdm.write(f"Loading Generative AI and VAE on {device}...")
    vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=dtype)
    pipe = StableDiffusionPipeline.from_pretrained(
        "SG161222/Realistic_Vision_V6.0_B1_noVAE", 
        vae=vae, 
        torch_dtype=dtype, 
        use_safetensors=False
    )
    pipe.safety_checker = None
    pipe.requires_safety_checker = False
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter-full-face_sd15.safetensors")
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    pipe.to(device)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if torch.cuda.is_available(): device, dtype = torch.device("cuda"), torch.float16
    elif torch.backends.mps.is_available(): device, dtype = torch.device("mps"), torch.float32
    else: device, dtype = torch.device("cpu"), torch.float32
        
    sd_pipe = setup_sd_pipeline(device, dtype)
    target_files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))], reverse=True)

    with tqdm(total=len(target_files), desc="Total Progress", position=0) as overall_pbar:
        for filename in target_files:
            criminal_id = os.path.splitext(filename)[0]
            person_dir = os.path.join(OUTPUT_DIR, criminal_id)
            
            if os.path.exists(person_dir) and len([f for f in os.listdir(person_dir) if f.endswith('.jpg')]) >= 11:
                overall_pbar.update(1); continue
            
            os.makedirs(person_dir, exist_ok=True)
            pil_img = Image.open(os.path.join(INPUT_DIR, filename)).convert("RGB")
            pil_img = pil_img.resize((512, 512)) 
            pil_img.save(os.path.join(person_dir, "00_original.jpg"))

            orig_gender = "man"
            
            # Use the new aggressive combinations
            tasks = build_disguise_tasks()

            csv_path = os.path.join(person_dir, "metadata.csv")
            write_header = not os.path.exists(csv_path)
            
            with open(csv_path, mode='a', newline='', encoding='utf-8') as csv_file:
                csv_writer = csv.writer(csv_file)
                if write_header:
                    csv_writer.writerow(["filename", "prompt", "ip_adapter_scale"])

                with tqdm(total=len(tasks), desc=f"ID: {criminal_id[:15]}", position=1, leave=False) as inner_pbar:
                    for i, task in enumerate(tasks):
                        
                        sd_pipe.set_ip_adapter_scale(task["scale"])
                        
                        # Forced "RAW photo" prefix ensures photorealism and clothing
                        final_prompt = f"RAW photo of a clothed {orig_gender} {task['prompt']}, highly detailed face, 8k uhd, dslr, professional photography"
                        
                        output_img = sd_pipe(
                            prompt=final_prompt,
                            ip_adapter_image=pil_img,
                            negative_prompt="deformed, mutated, ugly, bad anatomy, cloned face, disfigured, naked, shirtless, bare chest, nsfw, cartoon, blurry",
                            num_inference_steps=25,
                            guidance_scale=6.0  
                        ).images[0]
                        
                        if np.mean(np.array(output_img)) > 1.0:
                            gen_filename = f"gen_{i+1:02d}.jpg"
                            output_img.save(os.path.join(person_dir, gen_filename))
                            
                            csv_writer.writerow([gen_filename, final_prompt, task["scale"]])
                            csv_file.flush()
                        
                        inner_pbar.update(1)
            
            overall_pbar.update(1)

if __name__ == "__main__":
    main()

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading Generative AI and VAE on cuda...


C:\Users\kaksh\anaconda3\envs\newtorchenv\Lib\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


...

ID: interpol-red-20:   0%|          | 0/10 [00:00<?, ?it/s]

ID: interpol-red-20:   0%|          | 0/10 [00:00<?, ?it/s]

ID: interpol-red-20:   0%|          | 0/10 [00:00<?, ?it/s]

ID: interpol-red-20:   0%|          | 0/10 [00:00<?, ?it/s]

ID: interpol-red-20:   0%|          | 0/10 [00:00<?, ?it/s]